# 12 — Rebuilding HMM updates from memory (and a calculator)

Suppose you remember only three ingredients: an initial distribution $\pi_i$, transitions $A_{ij}$, and emissions $B_i(x)$. Reconstruct everything with one sentence:

> **Forward carries evidence in; backward carries future evidence back; multiply them to count what was probably hidden.**

Keep the index story fixed: $i$ is yesterday's state and $j$ is today's state.

## The five-line reconstruction

1. Start: $\alpha_1(j)=\pi_j B_j(x_1)$.
2. Move then observe: $\alpha_t(j)=B_j(x_t)\sum_i\alpha_{t-1}(i)A_{ij}$.
3. Future from here: $\beta_t(i)=\sum_j A_{ij}B_j(x_{t+1})\beta_{t+1}(j)$, with $\beta_T(i)=1$.
4. Occupancy: $\gamma_t(i)\propto\alpha_t(i)\beta_t(i)$.
5. Transition occupancy: $\xi_t(i,j)\propto\alpha_t(i)A_{ij}B_j(x_{t+1})\beta_{t+1}(j)$.

The memory check is temporal: `alpha` ends **here**, `beta` begins **after here**, and `xi` inserts the one bridge from here to next. Normalize $\gamma$ over states and $\xi$ over all state pairs.

## Turn probabilities into new parameters

If the hidden path were visible, you would count starts, arrows, and state/symbol pairs. With hidden states, use expected counts:

- $\pi_i^{new}=\gamma_1(i)$
- $A_{ij}^{new}=\frac{\sum_t\xi_t(i,j)}{\sum_j\sum_t\xi_t(i,j)}$
- $B_i(m)^{new}=\frac{\sum_{t:x_t=m}\gamma_t(i)}{\sum_m\sum_{t:x_t=m}\gamma_t(i)}$

So the whole M-step is: **expected counts, then row-normalize**. On paper, rescale every forward row to avoid tiny numbers and reuse those scale factors in the backward pass; with a calculator and a short sequence, normalizing each row is enough.

In [ ]:
import torch

from latentlinks.hmm import CategoricalHMM, baum_welch_step, forward_backward

model = CategoricalHMM(
    torch.tensor([0.6, 0.4]),
    torch.tensor([[0.7, 0.3], [0.2, 0.8]]),
    torch.tensor([[0.9, 0.1], [0.25, 0.75]]),
)
x = torch.tensor([0, 1, 1, 0])
p = forward_backward(model, x)
updated, _ = baum_welch_step(model, [x])
print("gamma =\n", p.gamma.round(decimals=4))
print("expected transition counts =\n", p.xi.sum(0).round(decimals=4))
print("new A =\n", updated.transition.round(decimals=4))
torch.testing.assert_close(p.gamma.sum(1), torch.ones(4, dtype=torch.float64))

### Emergency checklist

Draw arrows left-to-right; write an emission under the state being entered; check every updated row sums to one; expect likelihood not to decrease after a full EM step; and remember that swapping state names changes no model probability. If one check fails, fix indices before doing more arithmetic.